# Homework 12 — Your own molecule, in ORCA

**Due Monday Nov 23.** Open notes, open lecture, open internet.

In class we ran water and read what ORCA wrote. Do the same on a molecule you choose:
build its geometry, write the input, run `Opt Freq` **as a job on Swan**, bring the
files back here, and do the Lecture 23 analysis on them. The Learn pages for unit 11
have everything you need; this notebook only says what to do.

1. Run the **Setup** cell. It defines the functions from class. Edit `NAME`, nothing else.
2. Thirteen problems. Each stands on its own; if one gives you trouble, skip it.
3. Run the **Check in** cell and type its number into the CHEM 438 app.
4. Hand in a **zip** of this notebook and your ORCA files (section 7).

**Start the job early.** Some molecules take hours, and the queue adds its own wait.

<!-- c438-copy-note -->
> ### ▶️ First — make your own copy
> This notebook opened **read-only** (it comes straight from GitHub). To type your answers and keep your work:
>
> **File ▸ Save a copy in Drive**  —  or click **Copy to Drive** in the toolbar.
>
> Then work in *your* copy. It saves to your Google Drive automatically.

In [ ]:
# Setup — run this first (about a minute). Edit NAME, nothing else.
!pip install rdkit py3Dmol -q

import os
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem

def energies(out_file):
    """Every FINAL SINGLE POINT ENERGY in an ORCA output, in hartree, in order."""
    found = []
    for line in open(out_file):
        if "FINAL SINGLE POINT ENERGY" in line:
            found.append(float(line.split()[-1]))
    return found

def finished(out_file):
    return "ORCA TERMINATED NORMALLY" in open(out_file).read()

def frequencies(out_file):
    """The vibrational frequencies of an ORCA Freq run, in cm^-1; imaginary modes come out negative."""
    found = []
    inside = False
    for line in open(out_file):
        if "VIBRATIONAL FREQUENCIES" in line:
            inside = True
        elif "NORMAL MODES" in line:
            break
        elif inside and "cm**-1" in line:
            found.append(float(line.split()[1]))
    freqs = np.array(found)
    return freqs[freqs != 0]                   # the six zeros are translation and rotation, not vibrations

def verdict(out_file):
    """What the frequencies say about the geometry: minimum, transition state, or start again."""
    n_negative = np.sum(frequencies(out_file) < 0)
    if n_negative == 0:
        return "minimum"
    if n_negative == 1:
        return "transition state"
    return "start again"

def frames(trj_file):
    """Split a multi-frame .xyz into one string per frame."""
    lines = open(trj_file).read().splitlines()
    per_frame = int(lines[0]) + 2              # atoms + the two header lines
    result = []
    for start in range(0, len(lines), per_frame):
        result.append("\n".join(lines[start:start + per_frame]))
    return result

def coords(xyz_text):
    """(atom symbols, N x 3 array in Å) from one frame of .xyz text."""
    lines = xyz_text.splitlines()
    n_atoms = int(lines[0])
    atoms = []
    xyz = np.zeros((n_atoms, 3))
    for i, line in enumerate(lines[2:2 + n_atoms]):
        parts = line.split()
        atoms.append(parts[0])
        xyz[i] = parts[1:4]
    return atoms, xyz

def distance(a, b):
    """The distance between two points, in Å."""
    return np.linalg.norm(a - b)

def angle(a, b, c):
    """The angle at b, in degrees."""
    u = a - b
    w = c - b
    cos_theta = np.dot(u, w) / (np.linalg.norm(u) * np.linalg.norm(w))
    return np.degrees(np.arccos(cos_theta))

def show(xyz_file, width=400, height=300):
    """A 3D view of an .xyz file. The returned view must be the last line of the cell to appear."""
    v = py3Dmol.view(width=width, height=height)
    v.addModel(open(xyz_file).read(), "xyz")
    v.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
    v.zoomTo()
    return v

NAME = "aspirin"      # <- one lowercase word, no spaces: every file name below is built from it
print("ready; files will be called", NAME + "_opt_freq.*")

## 1. Pick a molecule

Any of these, or one of your own with **fewer than 100 atoms** counting hydrogens (ask
first if it is charged, has a metal, or is bigger than about 40 atoms). The last column is
roughly how long `Opt Freq` at B3LYP/def2-SVP takes on one core of Swan.

| molecule | SMILES | atoms | one core, `Opt Freq` |
|---|---|---|---|
| ethanol | `CCO` | 9 | under 2 min |
| urea | `NC(N)=O` | 8 | under 2 min |
| benzoic acid | `OC(=O)c1ccccc1` | 15 | about 10 min |
| adenine | `Nc1ncnc2[nH]cnc12` | 15 | about 10 min |
| vanillin | `COc1cc(C=O)ccc1O` | 19 | about half an hour |
| paracetamol | `CC(=O)Nc1ccc(O)cc1` | 20 | about half an hour |
| aspirin | `CC(=O)Oc1ccccc1C(=O)O` | 21 | 37 min |
| dopamine | `NCCc1ccc(O)c(O)c1` | 22 | about an hour |
| caffeine | `Cn1cnc2c1c(=O)n(C)c(=O)n2C` | 24 | about an hour |
| glucose | `OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O` | 24 | about an hour |
| serotonin | `NCCc1c[nH]c2ccc(O)cc12` | 25 | 1–2 hours |
| nicotine | `CN1CCC[C@H]1c1cccnc1` | 26 | 1–2 hours |
| limonene | `CC1=CC[C@H](CC1)C(=C)C` | 26 | 1–2 hours |
| menthol | `CC(C)[C@@H]1CC[C@@H](C)C[C@H]1O` | 31 | 2–3 hours |
| naproxen | `COc1ccc2cc(ccc2c1)[C@H](C)C(O)=O` | 31 | 2–3 hours |
| ibuprofen | `CC(C)Cc1ccc(cc1)C(C)C(O)=O` | 33 | 2–3 hours |

**Problem 1.**
1. Build your molecule from its SMILES with RDKit, hydrogens included, and relax it with the force field.
2. Write it to `NAME + ".xyz"`.
3. Print the number of atoms and `show` it.

In [ ]:
# Problem 1

**Problem 2.**
1. Count the electrons in your molecule and print the total.
2. In a `#` comment: what charge and multiplicity you will give ORCA, and why those two numbers are right for your molecule.

In [ ]:
# Problem 2

## 2. The input file, and the job on Swan

**Problem 3.**
1. Write the ORCA input for `Opt Freq` on your molecule, at B3LYP D3BJ def2-SVP on one core, with the geometry read from your `.xyz` file.
2. Save it as `NAME + "_opt_freq.inp"` and print it back.

In [ ]:
# Problem 3

Get the `.xyz` and the `.inp` onto Swan, into a folder of their own. The job script is the
one from class with the input's name as its argument:

```
wget https://raw.githubusercontent.com/jyesselm/chem438-notebooks/main/qm-orca/bigger/run_orca.sh
sbatch run_orca.sh NAME_opt_freq
```

**Problem 4.** As `#` comments in the cell:
1. The `Submitted batch job` line.
2. One `squeue` line from while it was waiting or running.
3. The `TOTAL RUN TIME` line from the finished `.out`.

In [ ]:
# Problem 4

## 3. Bring the files back

Three files matter: `NAME_opt_freq.out`, `NAME_opt_freq.xyz`, `NAME_opt_freq_trj.xyz`.
Get them from Swan to your laptop and upload them here. Keep the laptop copies.

**Problem 5.**
1. Print whether each of the three files exists.
2. Print whether the run finished.
3. In a comment: what a job killed by the time limit would show here, and why you check this before anything else.

In [ ]:
# Problem 5

## 4. Energies

**Problem 6.**
1. Print how many optimisation cycles there were.
2. Print the first and last energies.
3. Print the drop from first to last in kJ/mol, one decimal.
4. In a comment: water fell 53 kJ/mol from a hand drawing. More or less here, and why?

In [ ]:
# Problem 6

**Problem 7.**
1. Plot the energy above the final energy, in kJ/mol, against cycle number, with markers.
2. Label both axes with units and title it with your molecule's name.
3. In a comment: which cycle did most of the work?

In [ ]:
# Problem 7

## 5. Geometry

**Problem 8.**
1. Show the optimised structure.
2. In a comment: what, if anything, looks different from the starting structure in Problem 1.

In [ ]:
# Problem 8

**Problem 9.**
1. Print the atom symbols with their indices.
2. Print one bond length (three decimals) and one bond angle (one decimal) from the optimised structure.
3. In a comment: the textbook value for each, with its source, and how far off you are in per cent.

In [ ]:
# Problem 9

**Problem 10.**
1. From the trajectory, take the first and last frames.
2. Print the root-mean-square distance the atoms moved between them, three decimals.
3. Print the largest single move and which atom made it.
4. In a comment: is that consistent with Problem 8?

In [ ]:
# Problem 10

## 6. Frequencies

**Problem 11.**
1. Print how many vibrational frequencies there are, and how many there should be for your molecule.
2. Print how many are negative, and the verdict.
3. In a comment: what one negative frequency would mean, what two would, and what you would do in each case.

In [ ]:
# Problem 11

**Problem 12.**
1. Print the three highest frequencies.
2. Print how many frequencies are above 2800 cm⁻¹, and how many hydrogens your molecule has.
3. In a comment: should those two numbers match, do they, and which bond is the very highest one?

In [ ]:
# Problem 12

**Problem 13.** Pick one vibration in your molecule that has a measured infrared value you can look up.
1. Print the computed frequencies in a window around it.
2. Print the computed value next to the measured one, with the error in per cent.
3. Scale the computed value by 0.96 and print the error again.
4. In a comment: where the measured number came from. A number with no source is a guess.

In [ ]:
# Problem 13

## Check in

Run the cell below. It prints one number. **Type it into the CHEM 438 app to mark this
homework complete.** If it is not 0, your structure is not a minimum: see Problem 11,
rerun, and hand in the run that is.

In [ ]:
check = frequencies(NAME + "_opt_freq.out")
print("Your check-in value:", int(np.sum(check < 0)))

## 7. Hand it in

One zip file on Canvas, named `hw12_yourlastname.zip`, holding:

- this notebook, run top to bottom (**File ▸ Download ▸ Download .ipynb**)
- `NAME.xyz` and `NAME_opt_freq.inp`
- `NAME_opt_freq.out`, `NAME_opt_freq.xyz`, `NAME_opt_freq_trj.xyz`